# Módulo 16: Aplicación de la analítica — Guía del ejercicio

En este ejercicio vas a diseñar la propuesta de un equipo de ciencia de datos para una startup de pódcast: elegir y priorizar sus aplicaciones, definir los roles del equipo y decidir cómo medir el éxito. Después pondrás en práctica tres de esas aplicaciones con datos de escucha: métricas de *engagement*, una recomendación simple y la evaluación de una prueba A/B.

- Si necesitas repasar los conceptos, consulta la lección [m16-aplicacion-de-analitica-leccion.md](m16-aplicacion-de-analitica-leccion.md).
- Cuando termines, compara tu trabajo con la solución: [m16-aplicacion-de-analitica-solucion.ipynb](m16-aplicacion-de-analitica-solucion.ipynb).

## Contexto

**Resonar** es una startup de pódcast en español con unas 40 personas. Produce y licencia programas de crimen, negocios y tecnología, ciencia, humor, deportes y cultura, y se financia con una suscripción sin anuncios y con publicidad en el plan gratuito. Acaba de cerrar una ronda de inversión y los inversores le piden dos cosas: **que los usuarios nuevos se queden** y **que las decisiones de producto se tomen con datos**.

Hasta ahora, un único analista preparaba informes a mano. La dirección quiere formar un equipo de ciencia de datos pequeño y te pide una propuesta.

Como primera prueba, el equipo de producto lanzó un experimento con la función **"Cola automática"**: al terminar un episodio empieza el siguiente que el sistema recomienda, en lugar de detenerse. Los 3 000 usuarios que se registraron entre el 2 y el 8 de marzo de 2026 se asignaron al azar, mitad y mitad, al grupo **A** (control, sin la función) y al grupo **B** (con la función), y se registró su escucha hasta el 5 de abril.

## Datos

La celda inicial genera el DataFrame `eventos`, con una fila por sesión de escucha de un episodio:

| Columna | Descripción |
| --- | --- |
| `usuario_id` | Identificador del usuario |
| `contenido` | Nombre del programa escuchado |
| `categoria` | Categoría del programa |
| `fecha` | Fecha de la sesión |
| `minutos` | Minutos escuchados en la sesión |
| `grupo` | Grupo del experimento: `A` (control) o `B` (con cola automática) |

Cada usuario escuchó al menos un episodio el día en que se registró, así que **la fecha de su primer evento es su fecha de alta** y los 3 000 usuarios aparecen en la tabla. El DataFrame `catalogo` lista los 15 programas con su categoría. Los datos son sintéticos y se generan con una semilla fija, así que siempre obtendrás los mismos valores.

## Instrucciones

- Trabaja en este notebook (`uv run jupyter lab` desde la raíz del repositorio) y ejecuta primero la celda inicial.
- Las tareas 1 a 3 son preguntas escritas sobre el caso: responde en la celda de texto que sigue a cada una. Usa listas o tablas para que la propuesta sea fácil de leer.
- Las tareas 4 a 6 son de Python: escribe tu código en las celdas vacías y añade más celdas si lo necesitas.
- Acompaña cada resultado numérico con una interpretación breve en términos del negocio.

La celda siguiente importa las librerías y genera los datos.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

catalogo = pd.DataFrame({
    "contenido": ["Crimen sin Resolver", "Expedientes del Sur", "Casos Fríos",
                  "Startups al Día", "Código Abierto", "Finanzas en Corto",
                  "Ciencia de Bolsillo", "El Universo en 20 Minutos", "Mente Curiosa",
                  "Risas de Sobremesa", "Comedia sin Guion",
                  "La Jugada", "Tiempo Extra",
                  "Historias de Abuela", "Viajes Lentos"],
    "categoria": ["Crimen"] * 3 + ["Negocios y tecnología"] * 3 + ["Ciencia"] * 3
                 + ["Humor"] * 2 + ["Deportes"] * 2 + ["Cultura"] * 2,
})
categorias = catalogo["categoria"].unique()
prob_categoria = [0.25, 0.22, 0.18, 0.15, 0.12, 0.08]
todos = catalogo["contenido"].to_numpy()

n_usuarios = 3000
inicio, fin = pd.Timestamp("2026-03-02"), pd.Timestamp("2026-04-05")  # inicio es lunes
grupos = rng.permutation(np.repeat(["A", "B"], n_usuarios // 2))
dia_alta = rng.integers(0, 7, n_usuarios)  # días desde el 2 de marzo

filas = []
for uid in range(n_usuarios):
    # Programas favoritos: dos de una categoría preferida, más algunos hábitos cruzados
    cat = rng.choice(categorias, p=prob_categoria)
    propios = catalogo.loc[catalogo["categoria"] == cat, "contenido"].to_numpy()
    favoritos = list(rng.choice(propios, size=2, replace=False))
    if "Startups al Día" in favoritos and "Código Abierto" not in favoritos and rng.random() < 0.7:
        favoritos.append("Código Abierto")
    if "Crimen sin Resolver" in favoritos and rng.random() < 0.5:
        favoritos.append("Historias de Abuela")

    actividad = rng.beta(2, 4)
    decaimiento = 0.045 if grupos[uid] == "A" else 0.040
    for d in range((fin - inicio).days - dia_alta[uid] + 1):
        dia = dia_alta[uid] + d
        fin_de_semana = dia % 7 >= 5
        p = actividad * np.exp(-decaimiento * d) * (0.6 if fin_de_semana else 1.0)
        if d == 0 or rng.random() < p:
            for _ in range(1 + rng.poisson(0.5)):
                prog = rng.choice(favoritos) if rng.random() < 0.8 else rng.choice(todos)
                filas.append((uid + 1, prog, dia, round(float(rng.gamma(2.5, 9)), 1), grupos[uid]))

eventos = pd.DataFrame(filas, columns=["usuario_id", "contenido", "dia", "minutos", "grupo"])
eventos["fecha"] = inicio + pd.to_timedelta(eventos["dia"], unit="D")
eventos = eventos.merge(catalogo, on="contenido")
eventos = eventos[["usuario_id", "contenido", "categoria", "fecha", "minutos", "grupo"]]
eventos = eventos.sort_values(["fecha", "usuario_id"], ignore_index=True)
eventos.head()

## Tarea 1. Aplicaciones priorizadas

1. Propón **cinco aplicaciones** de la ciencia de datos para Resonar (por ejemplo, recomendación de episodios, predicción de abandono, segmentación para publicidad, decisiones de qué programas producir).
2. Valora cada una de 1 a 5 en **impacto** (cuánto mueve la retención o los ingresos) y en **viabilidad** (datos disponibles, dificultad técnica y tiempo para una startup de 40 personas).
3. Elige las **tres** que harías primero y justifica el orden.

*Escribe aquí tu respuesta.*

## Tarea 2. Composición del equipo

Propón un equipo inicial de **4 a 6 personas** para atender las tres aplicaciones elegidas. Para cada rol indica:

1. Sus responsabilidades principales.
2. Con qué áreas de Resonar trabaja (producto, contenido, publicidad, ingeniería…).
3. En qué orden contratarías los roles y por qué.

Ten en cuenta que, a diferencia de una gran empresa, aquí una misma persona puede cubrir varias funciones.

*Escribe aquí tu respuesta.*

## Tarea 3. Cómo medir el éxito

1. Elige una **métrica principal** para la empresa y justifica por qué refleja el valor que reciben los usuarios.
2. Para cada una de las tres aplicaciones de la Tarea 1, define una métrica de éxito y una **métrica de control** (*guardrail*) que no debería empeorar.
3. Explica cómo comprobarías que una mejora se debe a la aplicación y no a otros factores.
4. Indica cómo medirías el éxito del propio equipo de ciencia de datos al cabo de un año.

*Escribe aquí tu respuesta.*

## Tarea 4. Métricas de engagement

1. Explora `eventos`: número de filas, usuarios únicos por grupo y rango de fechas.
2. Calcula los **usuarios activos diarios** (DAU, *daily active users*): el número de usuarios distintos con al menos una sesión cada día. Represéntalos en un gráfico de líneas.
3. Calcula los **minutos por usuario activo** de cada día (minutos totales del día entre DAU del día) y su media en días laborables y en fines de semana.
4. Muestra los cinco programas con más minutos escuchados y cuántos oyentes distintos tiene cada uno.
5. Describe la evolución del DAU y el patrón semanal. ¿Por qué el DAU de esta cohorte no basta por sí solo para saber si el producto mejora?

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Tarea 5. Recomendación por coocurrencia

Vas a construir la regla "quienes escucharon X también escucharon Y".

1. Crea una matriz usuario × programa con 1 si el usuario escuchó el programa al menos una vez y 0 si no.
2. Calcula la matriz de **coocurrencia** programa × programa: cuántos usuarios escucharon ambos programas. (Pista: el producto de la matriz traspuesta por la matriz.)
3. Para un programa $X$ y otro $Y$, calcula:
   - la **confianza** $P(Y \mid X) = \dfrac{n(X \cap Y)}{n(X)}$, la proporción de oyentes de $X$ que también escucharon $Y$;
   - el **lift** $= \dfrac{P(Y \mid X)}{P(Y)}$, donde $P(Y)$ es la proporción de todos los usuarios que escucharon $Y$. Un lift mayor que 1 indica que la asociación es más fuerte de lo que explicaría la popularidad de $Y$.
4. Escribe una función `recomendar(programa, k=3)` que devuelva los $k$ programas con mayor confianza, junto con su lift.
5. Aplícala a `"Startups al Día"` y a `"Crimen sin Resolver"`. ¿Hay alguna recomendación que salga de la categoría del programa? ¿Por qué conviene mirar el lift además de la confianza?

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Tarea 6. Prueba A/B de la cola automática

Define la **retención en la semana 4** como haber tenido al menos una sesión entre los días 21 y 27 contados desde la fecha de alta (el día de alta es el día 0).

1. Calcula la fecha de alta de cada usuario y los días transcurridos desde el alta en cada evento.
2. Calcula, por grupo, el número de usuarios, cuántos retuvieron y la proporción de retención.
3. Plantea las hipótesis y aplica una prueba z de diferencia de proporciones con `proportions_ztest` de `statsmodels.stats.proportion`, con nivel de significación $\alpha = 0.05$.
4. Calcula el **tamaño del efecto**: la diferencia en puntos porcentuales, el aumento relativo y un intervalo de confianza del 95 % para la diferencia (puedes usar `confint_proportions_2indep`).
5. Como métrica de control, compara los minutos escuchados por usuario en cada grupo durante los primeros 28 días (días 0 a 27).
6. Interpreta el p-valor y el tamaño del efecto. ¿Recomendarías lanzar la función a todos los usuarios? ¿Qué riesgos o límites tiene la conclusión?

In [ ]:
# Tu código aquí

*Escribe aquí tu interpretación.*

## Antes de terminar

- [ ] Las tres aplicaciones elegidas están priorizadas con criterios de impacto y viabilidad.
- [ ] Cada rol del equipo tiene responsabilidades concretas y un orden de contratación.
- [ ] Cada aplicación tiene una métrica de éxito y una de control.
- [ ] El DAU cuenta usuarios distintos, no sesiones.
- [ ] La retención usa como denominador a todos los usuarios de cada grupo.
- [ ] La conclusión de la prueba A/B habla del p-valor **y** del tamaño del efecto.

Cuando termines, compara tu trabajo con la solución: [m16-aplicacion-de-analitica-solucion.ipynb](m16-aplicacion-de-analitica-solucion.ipynb).